In [2]:
import re
import itertools
from typing import Any
from sqlalchemy import select
from dataclasses import dataclass, field
from rapidfuzz.distance import Levenshtein
from typing import Literal
from collections import defaultdict

from database import fill_wiktionary, session_factory, EnglishWiktionary, RussianWiktionary
from settings import settings
from logger import logger

In [3]:
@dataclass
class Word:
    """Individual unit of the translation."""
    spelling: str
    transcription: str | None

@dataclass
class Translation:
    """Individual translation with its linguistic & phonosemantic properties."""
    words: list[Word]
    score: float | None = None
    phonotypes: tuple[str, ...] | None = None
    model: tuple[str, ...] | None = None

@dataclass
class Example:
    """Example sentance representing this meaning."""
    russian: str
    english: str

@dataclass
class Sense:
    """Meaning that can be translated in several ways."""
    translations: list[Translation]
    examples: list[Example]

@dataclass
class LexicalUnit:
    """NOMINAL word from user input."""
    lemma: str
    pos: Literal['verb', 'noun']
    onomatop_type: Literal['instant', 'continuant']
    senses: list[Sense] = field(default_factory=list)

    @classmethod
    def from_wiktionary(cls,
        lemma: str,
        pos: Literal['verb', 'noun'],
        onomatop_type: Literal['instant', 'continuant']
    ):
        instance = cls(lemma=lemma, pos=pos, onomatop_type=onomatop_type)
        entry = instance._get_russian_wiktionary_entry()
        if entry: instance._set_wiktionary_translation(entry=entry)
        return instance

    def _get_russian_wiktionary_entry(self) -> dict[str, Any] | None:
        """Get wiktionary entry for this Russian word by its lemma and POS."""
        with session_factory() as session:
            stmt = select(RussianWiktionary.entry).where(RussianWiktionary.word == self.lemma)
            if self.pos: stmt.where(RussianWiktionary.pos == self.pos)
            return session.scalars(stmt).first()

    @staticmethod
    def _get_english_wiktionary_entry(lemma: str, pos: str | None = None) -> dict[str, Any] | None:
        """Get wiktionary entry for Russian word by lemma and POS."""
        with session_factory() as session:
            stmt = select(EnglishWiktionary.entry).where(EnglishWiktionary.word == lemma)
            if pos: stmt.where(EnglishWiktionary.pos == pos)
            return session.scalars(stmt).first()

    def _set_wiktionary_translation(self, entry: dict[str, Any]):
        """Get entry from wiktionary."""
        # Loop meanings
        for sense in entry['senses']:
            # Loop various translations of each meaning
            translations: list[Translation] = []
            for link in sense['links']:
                # Loop links(words) of each translation
                words: list[Word] = []
                # Skip link if its invalid (link consists of two elements)
                if len(link) < 2: continue
                for word in link[1].split(' '):
                    trascription = self._get_wiktionary_transcription(word)
                    words.append(Word(
                        spelling=word,
                        transcription=trascription,
                    ))
                translations.append(
                    Translation(
                        words=words,
                    )
                )
            # Find examples for each meaning
            examples: list[Example] = [
                Example(
                    russian=example.get('text', ''),
                    english=example.get('english', ''),
                )
                for example in sense.get('examples', [])
            ]
            self.senses.append(Sense(translations=translations, examples=examples))

    @staticmethod
    def _get_wiktionary_transcription(word: str) -> str | None:
        entry = LexicalUnit._get_english_wiktionary_entry(lemma=word)
        if not entry: return
        sounds: list[dict[str, Any]] = entry.get('sounds', [])
        
        # Try to find each pronunciation in a priority list
        for tag in settings.wiktionary_pronunciation_priority:
            for sound in sounds:
                if 'ipa' in sound and tag in sound.get('tags', []):
                    # Select phonemic
                    if '[' not in sound['ipa']: return sound['ipa']
        
        # Fallback 1: take untagged "default" pronounciation
        for sound in sounds:
            if 'ipa' in sound and not sound.get('tags'):
                # Select phonemic
                if '[' not in sound['ipa']: return sound['ipa']
        
        # Fallback 2: take the first ipa entry that was found
        for sound in sounds:
            if 'ipa' in sound:
                # Select phonemic
                if '[' not in sound['ipa']: return sound['ipa']

    def _calculate_onomatopoeic_score(self) -> None:
        """Perform phonosemantic processing of each translation."""
        onomatopoeic_module = Onomatopoeic()
        for sense in self.senses:
            for translation in sense.translations:
                # Concat individual word transcriptions into one
                common_transcription = ''.join([word.transcription for word in translation.words if word.transcription])
                # Skip translation if there are no transcriptions
                if not common_transcription: continue
                # Perform phonosemantic processing
                score, phonotypes, model = onomatopoeic_module.weigh_transcription(
                    ipa=common_transcription,
                    type=self.onomatop_type
                )
                # Set score, phonotypes models & chosen model
                translation.score = score
                translation.phonotypes = phonotypes
                translation.model = model

In [6]:
class Onomatopoeic:
    """This is a class for onomatopoeic operations."""

    CONS = {
        'PLOS▼', 'PLOS▲', 'FRIC▼', 'FRIC▲',
        'AFFR▼', 'AFFR▲', 'SON_lat', 'SON_lab',
        'SON_nas', 'SON_med', 'SON_gutt', 'R',
    }
    PLOS = {'PLOS▼', 'PLOS▲'}
    AFFR = {'AFFR▼', 'AFFR▲'}
    R = {'R'}

    VOC_SHORT = {'VOC_l_w', 'VOC_h_w'}
    VOC_LONG = {'VOC_l_s', 'VOC_h_s'}

    EMPTY = {None}
    
    # Dictionary of all onomatopoeic models by class
    onomatopoeic_models = {
        # Instant
        "I": (
            # P.47
            (
                PLOS | AFFR,
                VOC_SHORT,
                PLOS,
            ),
            # P.48
            (
                PLOS | AFFR,
                VOC_SHORT,
            ),
        ),
        # TODO: Уточнить, что именно у Воронина долгий, а что короткий
        # Tonal Continuant
        "TC": (
            # TODO: В тексте написано, что VOC_LONG. Выглядит как VOC_SHORT. Выяснить причину
            # P.49 (53)
            (
                CONS,
                {'SON_lat', 'SON_lab', None},
                VOC_LONG,
                PLOS | EMPTY,
            ),
            # P.49 (53) NOTE: Variation of the above
            (
                CONS | EMPTY,
                VOC_LONG,
                PLOS | EMPTY,
            ),
        ),
        # Noisy Continuant
        "NC": (
            # P.52 (56)
            (
                {'FRIC▲'},
                VOC_SHORT,
                CONS | EMPTY,
            ),
            # P.52 (56) NOTE: variation of the above
            (
                CONS | EMPTY,
                VOC_SHORT,
                {'FRIC▲'},
            ),
        ),
        # Tonal Noisy Continuant
        "TNC": (
            # P.53 (57)
            (
                CONS,
                VOC_SHORT,
                {'FRIC▼'},
            ),
        ),
        # Frecventatives
        "F": (
            # P.54 (58) TODO: Непонятные стрелочки на схеме
            (
                CONS | EMPTY,
                R,
                VOC_SHORT,
                PLOS,
            ),
            # P.56 (60)
            (
                CONS,
                VOC_SHORT,
                R,
            ),
            # P.57 (61)
            (
                CONS,
                VOC_SHORT,
                R,
            ),
            # P.57 (61)
            (
                CONS,
                R,
                VOC_LONG,
                CONS | EMPTY,
            ),
            # P.58 (62) NOTE: Included in #56
            # (
            #     {'FRIC▲'},
            #     VOC_SHORT,
            #     R,
            # ),
            # P.58 (62)
            (
                R,
                VOC_SHORT,
                {'FRIC▲'},
            ),
            # P.59 (62) TODO: Непоняные стрелочки на схеме
            (
                {'FRIC▲', None},
                R,
                VOC_SHORT,
                {'FRIC▼'},
            ),
        ),
        # "": (

        # )
    }

    phonotypes = {
        # 24 сonsonant phonemes
        # PLOS (6)
        'PLOS▼' : ('b', 'd', 'g'),
        'PLOS▲' : ('t', 'p', 'k'),
        # FRIC (8) (sib▲▼?)
        'FRIC▼': ('v', 'z', 'ð', 'ʒ'),
        'FRIC▲': ('f', 's', 'θ', 'ʃ'),
        # AFFR (2)
        'AFFR▼': ('d͡ʒ',) + ('dʒ',),
        'AFFR▲': ('t͡ʃ',) + ('tʃ',),
        # SON (7)
        'SON_lat': ('l',),  # боковые
        'SON_lab': ('m', 'w'),  # губные
        'SON_nas': ('m', 'n', 'ŋ'), # назальные
        'SON_med': ('j'),  # среднеязычные
        'SON_gutt': ('h', 'ŋ'),  # заднеязычные и фарингальные
        # R (1)
        'R': ('r',) + ('ɹ',),  # вибрант

        # 12(+1) monophthongs, 8(+1) diphthongs vowel phonemes (N.B. /ɛ/ & /oʊ/ not by Bondarko)
        'VOC_h_w': ('ɪ', 'ʊ', 'ə', 'e', 'ɛ'),
        'VOC_h_s': ('i', 'u', 'ɪə', 'ʊə', 'eɪ','əʊ', 'oʊ'),
        'VOC_l_w': ('æ', 'ʌ'),
        'VOC_l_s': ('ɑ', 'ɒ', 'ɔ', 'ɜ', 'aʊ', 'aɪ', 'ɔɪ', 'ɛə') + ('ɝ',),
    }

    # Reverse dict[phoneme, list[phonotype]] for tokenization
    phonotype_dict: dict[str, list[str]]
    # Sorted phonemes for maximal munch (complex phonemes first)
    sorted_phonemes: list[str]

    def __init__(self):
        # Create a dict of phonemes
        self.phonotype_dict = Onomatopoeic._collect_phonotypes_by_phoneme()
        # Sort phonemes: complex phonemes come first (maximal munch)
        self.sorted_phonemes = sorted(
            Onomatopoeic._collect_phonotypes_by_phoneme(), key=len, reverse=True
        )

    @classmethod
    def _model_variations(cls, type: Literal['instant', 'continuant']) -> tuple[tuple[str, ...], ...]:
        """Get all possible canonical model variations for input onomatopoeic class."""
        return tuple(
            tuple(elem for elem in variant if elem is not None)
            for model in cls.onomatopoeic_models[type]
            for variant in itertools.product(*(sorted(s, key=lambda x: (x is None, x)) for s in model))
        )

    @classmethod
    def _collect_phonotypes_by_phoneme(cls) -> dict[str, list[str]]:
        """Make a dict where keys are phonemes & values are their phonotypes."""
        phoneme_dict = defaultdict(list)
        for group, phonemes in cls.phonotypes.items():
            for phoneme in phonemes:
                phoneme_dict[phoneme].append(group)
        return phoneme_dict

    def _transcription_to_phonemes(self, ipa: str) -> list[str]:
        """Turn input transcription into a list of verified phonemes."""
        # Normalize transcription
        ipa = re.sub(r'[/\[\]\',ː:.]', '', ipa)
        # Verify phonemes
        tokens: list[str] = []
        i = 0
        while i < len(ipa):
            matched = next((ph for ph in self.sorted_phonemes if ipa.startswith(ph, i)), None)
            if matched:
                tokens.append(matched)
                i += len(matched)
            else:
                logger.error(f'Unknown token found: {ipa[i]} in {ipa}')
                i += 1
        return tokens

    def _transcription_to_phonotypes(self, ipa: str, ) -> list[tuple[str, ...]]:
        """Find every way to describe phonemic transcription with phonotypes."""
        phonemes = self._transcription_to_phonemes(ipa)
        phonotype_groups = [
            self.phonotype_dict[phoneme]
            for phoneme in phonemes
        ]
        # Find all possible ways to tokenize transcription
        return list(itertools.product(*phonotype_groups))

    def weigh_transcription(self,
            ipa: str,
            type: Literal['instant', 'continuant']
        ) -> tuple[float, tuple | None, tuple | None]:
        """Weigh ipa transcription to get its phonosemantic score & phonotypes."""
        best_score = 0.0
        best_phonotypes_variant = None
        best_model_variant = None
        for phonotypes_variant in self._transcription_to_phonotypes(ipa):
            for model_variant in Onomatopoeic._model_variations(type):
                score = Levenshtein.normalized_similarity(model_variant, phonotypes_variant)
                # print(f'{model_variant} <-> {phonotypes_variant}: {score}')
                if score > best_score:
                    best_score = score
                    best_phonotypes_variant = phonotypes_variant
                    best_model_variant = model_variant
                    if best_score >= 1.0:
                        return round(best_score, 2), best_phonotypes_variant, best_model_variant
        return round(best_score, 2), best_phonotypes_variant, best_model_variant

In [25]:
Onomatopoeic._model_variations('cont_tonal')

KeyError: 'cont_tonal'

In [7]:
def phonosemantic_translation(word: str, pos: str, onomatop_type: str):
    original = LexicalUnit.from_wiktionary(word, pos, onomatop_type)
    original._calculate_onomatopoeic_score()

    print(original)

In [8]:
phonosemantic_translation('бить', 'verb', 'I')

LexicalUnit(lemma='бить', pos='verb', onomatop_type='I', senses=[Sense(translations=[Translation(words=[Word(spelling='beat', transcription='/biːt/')], score=0.67, phonotypes=('PLOS▼', 'VOC_h_s', 'PLOS▲'), model=('PLOS▼', 'VOC_h_w', 'PLOS▲'))], examples=[Example(russian='В на́ше вре́мя учи́тель никогда́ не бьёт ученика́.', english='In our time the teacher never beats the student.'), Example(russian='Ты ду́маешь, меня́ не би́ли? Меня́, Олё́ша, так би́ли, что ты э́того и в стра́шном сне не уви́дишь.', english='Do you think I was never thrashed? I got such beatings, the like you’d never see even in a nightmare.')]), Sense(translations=[Translation(words=[Word(spelling='chime', transcription='/tʃaɪm/')], score=0.33, phonotypes=('AFFR▲', 'VOC_l_s', 'SON_lab'), model=('AFFR▲', 'VOC_h_w', 'PLOS▲'))], examples=[Example(russian='бить в ладо́ши', english='to clap (palms), causing applause'), Example(russian='Режиссёр име́ет привы́чку бить в ладо́ши что́бы привле́чь внима́ние.', english='The prod

In [ ]:
Onomatopoeic._model_variations('instant')

In [ ]:
# Fill DB table with jsonl data
fill_wiktionary(language='russian')